# Tech Challenge Fase 2  
## Bronze — Metas UFs

Este notebook realiza a ingestão dos arquivos Excel de **Metas UFs** para a camada Bronze.

A leitura segue o padrão adotado nos notebooks Silver enviados, utilizando `pandas.read_excel`.

### Entrada

```text
raw/metas_ufs/
├── metas_ufs_2023.xlsx
├── metas_ufs_2024.xlsx
└── metas_ufs_2025.xlsx
```

### Saída

```text
bronze/metas_ufs/
├── ano=2023/
├── ano=2024/
└── ano=2025/
```

## 1. Contexto na Arquitetura Medalhão

Este notebook atua exclusivamente na camada **Bronze**.

```text
Raw
 ↓
Bronze  ← você está aqui
 ↓
Silver
 ↓
Gold
```

A Bronze preserva os dados disponibilizados pelo INEP, acrescentando apenas metadados técnicos.

## 2. Pré-requisito técnico: openpyxl

O pandas utiliza a biblioteca `openpyxl` para leitura de arquivos `.xlsx`.

Caso o ambiente Databricks apresente erro de dependência, execute a célula abaixo uma vez no notebook.

In [0]:
%pip install openpyxl

## 3. Reinicialização opcional do Python

Após instalar uma biblioteca com `%pip`, o Databricks pode solicitar reinicialização do Python.


In [0]:
dbutils.library.restartPython()

## 4. Imports

In [0]:
import json
from datetime import datetime

import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType
)

## 5. Leitura do config.json

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(dbutils.fs.head(CONFIG_FILE_PATH))

BASE_PATH = config["environment"]["base_path"]
RAW_PATH = config["paths"]["raw_path"]
BRONZE_PATH = config["paths"]["bronze_path"]
LOG_PATH = config["paths"]["log_path"]
CONFIG_PATH = config["paths"]["config_path"]
EXECUTION_DATE = config["project"]["execution_date"]

print("BASE_PATH:", BASE_PATH)
print("RAW_PATH:", RAW_PATH)
print("BRONZE_PATH:", BRONZE_PATH)
print("CONFIG_PATH:", CONFIG_PATH)
print("EXECUTION_DATE:", EXECUTION_DATE)

## 6. Leitura da bronze_metadata para `metas_ufs`

In [0]:
metadata_path = f"{CONFIG_PATH}/bronze_metadata"

df_metadata = spark.read.parquet(metadata_path)

df_metadata_dataset = (
    df_metadata
    .filter(F.col("dataset") == "metas_ufs")
    .orderBy("ano")
)

display(df_metadata_dataset)

## 7. Validação dos metadados esperados

In [0]:
anos_esperados = [2023, 2024, 2025]

anos_metadata = [
    row["ano"] for row in df_metadata_dataset.select("ano").distinct().collect()
]

anos_faltantes = sorted(list(set(anos_esperados) - set(anos_metadata)))

if anos_faltantes:
    raise Exception(f"Metadados ausentes para os anos: {anos_faltantes}")
else:
    print("Metadados encontrados para todos os anos esperados.")

## 8. Função de leitura Excel com pandas

In [0]:
def read_excel_bronze(file_path: str, sheet_name: str, skip_rows: int):
    df_pandas = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        skiprows=skip_rows,
        engine="openpyxl",
        dtype=str
    )

    df_pandas = df_pandas.dropna(axis=1, how="all")
    df_pandas = df_pandas.dropna(axis=0, how="all")

    df_pandas.columns = [
        str(col).strip()
        .replace(" ", "_")
        .replace(".", "_")
        .replace("-", "_")
        for col in df_pandas.columns
    ]

    df_pandas = df_pandas.fillna("")

    for col in df_pandas.columns:
        df_pandas[col] = df_pandas[col].astype(str).str.strip()

    # Cria Spark DataFrame com schema string explícito
    schema = StructType([
        StructField(str(col), StringType(), True)
        for col in df_pandas.columns
    ])

    df_spark = spark.createDataFrame(
        df_pandas.astype(str).values.tolist(),
        schema=schema
    )

    return df_spark

## 9. Função de enriquecimento técnico da Bronze

In [0]:
def add_bronze_metadata(df, dataset, source_file, source_format, ano_referencia):
    return (
        df
        .withColumn("_dataset", F.lit(dataset))
        .withColumn("_source_file", F.lit(source_file))
        .withColumn("_source_format", F.lit(source_format))
        .withColumn("_ano_referencia", F.lit(int(ano_referencia)))
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_execution_date", F.lit(EXECUTION_DATE))
        .withColumn("_pipeline_step", F.lit("bronze_metas_ufs"))
    )

## 10. Execução da ingestão Bronze — Metas UFs

In [0]:
execution_logs = []

rows_metadata = df_metadata_dataset.collect()

for row in rows_metadata:
    dataset = row["dataset"]
    ano = int(row["ano"])
    file_name = row["file_name"]
    raw_path = row["raw_path"]
    bronze_output_path = row["bronze_path"]
    source_format = row["source_format"]
    sheet_name = row["sheet_name"]
    skip_rows = int(row["skip_rows"]) if row["skip_rows"] is not None else 0

    start_time = datetime.now()

    print("=" * 100)
    print(f"Iniciando ingestão Bronze — Dataset: {dataset} | Ano: {ano}")
    print(f"Arquivo origem: {raw_path}")
    print(f"Aba: {sheet_name}")
    print(f"Destino Bronze: {bronze_output_path}")

    try:
        df_raw = read_excel_bronze(
            file_path=raw_path,
            sheet_name=sheet_name,
            skip_rows=skip_rows
        )

        df_bronze = add_bronze_metadata(
            df=df_raw,
            dataset=dataset,
            source_file=raw_path,
            source_format=source_format,
            ano_referencia=ano
        )

        records_read = df_bronze.count()
        columns_count = len(df_bronze.columns)

        (
            df_bronze
            .coalesce(1)
            .write
            .mode("overwrite")
            .format("parquet")
            .option("compression", "snappy")
            .save(bronze_output_path)
        )

        end_time = datetime.now()

        execution_logs.append({
            "dataset": str(dataset),
            "ano": int(ano),
            "file_name": str(file_name),
            "source_path": str(raw_path),
            "target_path": str(bronze_output_path),
            "status": "SUCCESS",
            "records_read": int(records_read),
            "columns_count": int(columns_count),
            "start_time": start_time.isoformat(),
            "end_time": end_time.isoformat(),
            "error_message": ""
        })

        print(f"Ingestão concluída com sucesso para {dataset} {ano}")

    except Exception as e:
        end_time = datetime.now()

        execution_logs.append({
            "dataset": str(dataset),
            "ano": int(ano),
            "file_name": str(file_name),
            "source_path": str(raw_path),
            "target_path": str(bronze_output_path),
            "status": "FAILED",
            "records_read": 0,
            "columns_count": 0,
            "start_time": start_time.isoformat(),
            "end_time": end_time.isoformat(),
            "error_message": str(e)
        })

        print(f"Erro na ingestão do arquivo {file_name}: {e}")

## 11. Criação e persistência do log de execução

In [0]:
schema_execution_logs = StructType([
    StructField("dataset", StringType(), True),
    StructField("ano", IntegerType(), True),
    StructField("file_name", StringType(), True),
    StructField("source_path", StringType(), True),
    StructField("target_path", StringType(), True),
    StructField("status", StringType(), True),
    StructField("records_read", LongType(), True),
    StructField("columns_count", IntegerType(), True),
    StructField("start_time", StringType(), True),
    StructField("end_time", StringType(), True),
    StructField("error_message", StringType(), True)
])

df_execution_logs = spark.createDataFrame(
    execution_logs,
    schema=schema_execution_logs
)

display(df_execution_logs.orderBy("ano"))

log_output_path = f"{LOG_PATH}/pipeline_execution/bronze/metas_ufs_execution_date={EXECUTION_DATE}"

(
    df_execution_logs
    .coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(log_output_path)
)

print("Log salvo em:", log_output_path)

## 12. Validação da Bronze gerada

In [0]:
for row in rows_metadata:
    ano = int(row["ano"])
    bronze_output_path = row["bronze_path"]

    print("=" * 100)
    print(f"Validação Bronze — {row['dataset']} | ano={ano}")
    print(bronze_output_path)

    try:
        df_validacao = spark.read.parquet(bronze_output_path)
        print("Colunas:", len(df_validacao.columns))
        display(df_validacao.limit(5))
    except Exception as e:
        print(f"Erro ao validar Bronze ano={ano}: {e}")

## 13. Checklist final

In [0]:
falhas = df_execution_logs.filter(F.col("status") == "FAILED").count()

if falhas > 0:
    display(df_execution_logs.filter(F.col("status") == "FAILED"))
    raise Exception(f"Foram encontradas {falhas} falhas na ingestão Bronze.")
else:
    print("Checklist final concluído com sucesso.")

## Resultado esperado

A camada Bronze de `metas_ufs` deve estar criada por ano.

### Próximo notebook

```text
03_silver_processing_quality
```